In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from langchain_ollama import ChatOllama

/Users/prasanthravula/LLM_ENGINEERING_PROJECTS/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [ ]:
llm = ChatOllama(model="kimi-k2.5:cloud")
links = fetch_website_links("https://edwarddonner.com")

['https://edwarddonner.com/', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/proficient/', 'https://edwarddonner.com/connect-four/', 'https://edwarddonner.com/outsmart/', 'https://edwarddonner.com/about-me-and-about-nebula/', 'https://edwarddonner.com/posts/', 'https://edwarddonner.com/', 'https://news.ycombinator.com', 'https://nebula.io/?utm_source=ed&utm_medium=referral', 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html', 'https://edwarddonner.com/curriculum/', 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/', 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/', 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/', 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/', 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/', 

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [12]:
def select_relevant_links(url):
    response = llm.invoke(
            [
                {"role": "system", "content": link_system_prompt},
                {"role": "user", "content": get_links_user_prompt(url)}
            ]
    )   
    result = response.content.strip()
    # Strip markdown code fences if present (e.g. ```json ... ```)
    if result.startswith("```"):
        result = result.split("```")[1]
        if result.startswith("json"):
            result = result[4:]
        result = result.strip()
    links = json.loads(result)
    return links

select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company website',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'press release',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'},
  {'type': 'linkedin profile',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

### Second step: make the brochure!

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Updated
6 days ago
•
281k
•
1.58k
Updated
2 days ago
•
2.45k
•
458
Updated
2 days ago
•
20k
•
455
Updated
3 days ago
•
15.6k
•
579
Updated
19 days ago
•
519k
•
1.06k
Browse 2M+ models
Spaces
Wan2.2 14B Preview
🐌
1.68k
generate a video from an image with a text prompt
DLSS 5 Anything
🎮
297
Turn any image into a DLSS 5 meme (using FLUX.2-klein-9b-kv)
Free Unlimited Google Veo 3
🌖
407
Free Unlimited Google Veo-3 NSFW Uncensored
FireRed Image Edit 1.0 Fast
🌖
537
FireRed-Image-Edit × Qwen-Image-Edit-Rapid (Transformers)
Kimodo
🏃
129
Generate high-quality motions from text prompts
Browse 1M+ applications
Datasets
Updated
2 days ago
•
21.1k
•
220
Updated
3 minutes ago
•
13.8k
•
216
Updated
3 days ago
•
1.54k
•
65
Update

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nUpdated\n6 days ago\n•\n281k\n•\n1.58k\nUpdated\n2 days ago\n•\n2.45k\n•\n458\nUpdated\n2 days ago\n•\n20k\n•\n455\nUpdated\n3 days ago\n•\n15.6k\n•\n579\nUpdated\n19 days ago\n•\n519k\n•\n1.06k\nBrowse 2M+ models\nSpaces\nWan2.2 14B Preview\n🐌\n1.68k\ngenerate a video from an image with a text prompt\nDLSS 5 Anything\n🎮\n297\nTurn any image into a DLSS 5 meme (using FLUX.2-klein-9b-kv)\nFree Unlimited Google Veo 3\n🌖\n407\nFree Unlimited Google Veo-3 NSFW Uncensored\nFireRed Image E

In [16]:
def create_brochure(company_name, url):
    response = llm.invoke(
        [
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ]
    )
    display(Markdown(response.content))
    
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: The AI Community Building the Future

**Democratizing good machine learning, one commit at a time.**

Hugging Face is the world's leading platform where the machine learning community creates, discovers, and collaborates on AI models, datasets, and applications. With over 2 million models, 1 million applications, and 500,000 datasets, we serve as the central hub for the open-source AI ecosystem.

## Our Platform

**The Home of Machine Learning**
We provide the infrastructure for researchers, developers, and organizations to move faster with AI:

- **Unlimited Public Collaboration**: Host and collaborate on models, datasets, and applications at no cost
- **All Modalities**: Explore text, image, video, audio, and 3D machine learning
- **Portfolio Building**: Share your work with the world and establish your ML profile
- **Open Source Stack**: Accelerate development with our industry-standard tools

## Enterprise Solutions

**Team & Enterprise Plans**
Scale your organization with enterprise-grade security and support:

- **Team Plans**: Starting at $20/user/month with advanced collaboration features
- **Enterprise**: Flexible contracts including Single Sign-On (SSO), regional data management, comprehensive audit logs, resource groups, and dedicated priority support
- **Inference Providers**: Access 45,000+ models from leading AI providers through a single, unified API with no service fees
- **Compute Solutions**: Deploy on optimized Inference Endpoints or upgrade Spaces applications for production workloads

## Community & Culture

**180 Team Members, Millions of Contributors**
Our culture is rooted in openness, collaboration, and the belief that AI should be accessible to everyone. We are:

- **Community-First**: Active contributors to papers, datasets, and cutting-edge research (recent highlights include work on multimodal models like SmolVLM and analysis of the global open-source AI ecosystem)
- **Mission-Driven**: Committed to democratizing "good" machine learning—ethical, accessible, and beneficial AI
- **Globally Aware**: Publishing insights on the shifting compute landscape and the future of open-source AI from DeepSeek to AI+

## Join Us

We're always looking for passionate individuals who believe in democratizing AI. With active development across research, engineering, and community building, Hugging Face offers opportunities to work on the infrastructure powering the future of machine learning.

**Ready to build the future?** Visit our careers page to explore current openings and become part of the community shaping tomorrow's AI.

---

*For press inquiries and partnership opportunities, contact our team through the Hugging Face website.*

### Finally - a minor improvement
With a small adjustment, we can change this so that the results stream back from OpenAI, with the familiar typewriter animation

In [17]:
def stream_brochure(company_name, url):
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in llm.stream(
        [
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ]
    ):
        response += chunk.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)

# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face
## The AI Community Building the Future

Hugging Face is the premier platform where the machine learning community collaborates, creates, and shares the future of AI. From independent researchers to enterprise teams, we provide the open-source infrastructure and tools that power the global AI ecosystem.

### The Home of Machine Learning

At the heart of Hugging Face lies a thriving collaboration platform featuring:
- **2M+ Models** – State-of-the-art machine learning models for every modality and use case
- **1M+ Applications** – Interactive AI apps (Spaces) spanning video generation, image editing, motion synthesis, and more
- **500k+ Datasets** – Curated training data for research and production

We support all modalities—text, image, video, audio, and 3D—enabling true multimodal innovation through the HF Open Source Stack.

### Culture of Open Collaboration

Our culture is rooted in transparency, accessibility, and community-driven progress. We believe AI should be open and available to everyone, which is why we offer unlimited public hosting for models, datasets, and applications. The platform is designed to help you move faster while building your ML portfolio and sharing your work with the world.

The community is actively pushing boundaries, as seen in recent submissions on autonomous evolutionary search and visual representation alignment, alongside thriving projects like Diffusers and Transformers.

### Enterprise Solutions

For organizations scaling AI in production, Hugging Face Team & Enterprise provides:
- **Enterprise Security**: Single Sign-On, audit logs, and resource groups
- **Global Infrastructure**: Regional deployment options and priority support
- **Compute Solutions**: Optimized Inference Endpoints and scalable Spaces applications
- **Unified API**: Access 45,000+ models from leading AI providers with no service fees
- **Private Data Management**: Secure dataset viewers and access controls

Starting at $20/user/month, our enterprise platform combines the innovation of the open community with enterprise-grade security and dedicated support.

### Build Your Future in AI

Hugging Face is where ML careers are made. Researchers, engineers, and creators use our platform to showcase breakthrough work, collaborate on cutting-edge projects, and establish themselves in the AI community. Whether you're contributing to open-source or deploying enterprise solutions, you'll be working with the tools that define the future of artificial intelligence.

**Join the community** that's building the future. Create, discover, and collaborate at huggingface.co.